# Flight Trend & Optimization Analyzer

## 1. Project Overview

This project fetches **real-time global flight data** from the [OpenSky Network API](https://opensky-network.org) and performs data analysis using Object-Oriented Programming (OOP) in Python.

**Core objectives:**
1. Fetch live flight data via REST API
2. Analyze country distribution and airline activity
3. Study velocity and altitude patterns
4. Filter and export data for further use


---
## 2. Environment Setup & Module Import


In [ ]:
import pandas as pd
from src.flight_logic import FlightDataAnalyzer, LiveFlightAPI

print("[INFO] Environment initialized. Modules loaded: FlightDataAnalyzer, LiveFlightAPI.")


---
## 3. Load Live Data from OpenSky Network API

We use the `LiveFlightAPI` child class (inherits from `FlightDataAnalyzer`) to fetch real-time flight data.
The endpoint `https://opensky-network.org/api/states/all` delivers a dynamic **JSON response**:
- The root object contains `time` (Unix timestamp) and `states` (a 2D array of airborne aircraft state vectors).
- Because `states` contains an array of raw values rather than named keys, our ingestion logic parses the official index positions:
  - `flight[0]`: **icao24** (unique 24-bit transponder hex ID)
  - `flight[1]`: **callsign** (8-character flight identifier)
  - `flight[2]`: **country** (aircraft country of registration)
  - `flight[5]`: **longitude** & `flight[6]`: **latitude** (WGS-84 coordinates)
  - `flight[7]`: **altitude** (barometric altitude in meters)
  - `flight[9]`: **velocity** (ground speed in m/s)
  - `flight[10]`: **heading** (direction of motion in degrees)

The parsed records are converted into a `pandas.DataFrame` and cleaned of missing transponder records.


In [ ]:
# Initialize API loader and fetch live flight data
live_api = LiveFlightAPI()

if live_api.fetch_live_flights(limit=500):
    print(f"[SUCCESS] Loaded {len(live_api.df)} live flights from OpenSky Network")
    print(f"Columns: {list(live_api.df.columns)}")
    display(live_api.df.head(10))
else:
    print("[ERROR] Failed to fetch data. Check network connection.")


---
## 4. Feature 1: Country Distribution Analysis

Which countries have the most airborne flights right now?


In [ ]:
# Top 10 countries by live flight count (tabular analysis)
country_stats = live_api.df["country"].value_counts().head(10)

print("=== Top 10 Countries by Live Flight Count ===")
for country, count in country_stats.items():
    print(f"  {country}: {count} flights")


In [ ]:
live_api.show_country_stats()
